# Lesson 9: LLM-based NER with GPT-4, Claude, and Llama

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand how LLMs can be used for NER
2. Design effective prompts for entity extraction
3. Compare GPT-4, Claude, and open-source LLMs
4. Handle LLM-specific challenges (hallucination, consistency)
5. Build hybrid LLM+specialized model pipelines

---

## 📚 Table of Contents

1. [LLMs for NER: Overview](#1-llms-for-ner-overview)
2. [Prompt Engineering for NER](#2-prompt-engineering-for-ner)
3. [Using OpenAI GPT-4](#3-using-openai-gpt-4)
4. [Using Anthropic Claude](#4-using-anthropic-claude)
5. [Using Open-Source LLMs (Llama, Mistral)](#5-using-open-source-llms)
6. [Handling LLM Challenges](#6-handling-llm-challenges)
7. [LLM vs Specialized Models](#7-llm-vs-specialized-models)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q openai anthropic transformers torch

In [ ]:
# Import libraries
import json
import re
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

---

## 1. LLMs for NER: Overview

### Why Use LLMs for NER?

| Advantage | Description |
|-----------|-------------|
| **Zero-shot** | No training data required |
| **Flexible** | Any entity type via prompting |
| **Context-aware** | Deep semantic understanding |
| **Reasoning** | Can explain entity decisions |

### Challenges

| Challenge | Description |
|-----------|-------------|
| **Hallucination** | LLMs may invent entities |
| **Inconsistency** | Different outputs for same input |
| **Speed** | Slower than specialized models |
| **Cost** | API calls can be expensive |
| **Accuracy** | Often lower F1 than fine-tuned models |

### Research Findings

> Despite the fact that LLMs have achieved SOTA performances on various NLP tasks, their performance on NER is still significantly below supervised baselines. GPT-4 achieves ~71% F1 on biomedical NER vs ~90% for specialized models.
>
> — [GPT-NER Paper, 2023](https://arxiv.org/abs/2304.10428)

---

## 2. Prompt Engineering for NER

### Basic Prompt Structure

In [ ]:
# Basic NER prompt templates

BASIC_PROMPT = """
Extract all named entities from the following text.
Return the results as a JSON array with objects containing "text" and "type" fields.

Entity types to extract: {entity_types}

Text: {text}

Output:
"""

# More structured prompt with examples (few-shot)
FEW_SHOT_PROMPT = """
You are a Named Entity Recognition system. Extract entities from text.

Entity types:
- PERSON: Names of people
- ORGANIZATION: Companies, institutions, agencies
- LOCATION: Cities, countries, geographic locations
- DATE: Dates and time expressions

Examples:

Text: "Apple CEO Tim Cook announced new products in Cupertino."
Output: [{"text": "Apple", "type": "ORGANIZATION"}, {"text": "Tim Cook", "type": "PERSON"}, {"text": "Cupertino", "type": "LOCATION"}]

Text: "The United Nations met in Geneva on March 15, 2024."
Output: [{"text": "United Nations", "type": "ORGANIZATION"}, {"text": "Geneva", "type": "LOCATION"}, {"text": "March 15, 2024", "type": "DATE"}]

Now extract entities from:

Text: "{text}"
Output:
"""

# GPT-NER style prompt (reduces hallucination)
GPT_NER_PROMPT = """
Extract {entity_type} entities from the text below.
Mark entities by surrounding them with @@ and ##.

Text: {text}

Output the text with {entity_type} entities marked:
"""

print("📝 Prompt Templates Defined")

In [ ]:
# Advanced prompt with output format specification
STRUCTURED_PROMPT = """
You are an expert Named Entity Recognition system.

## Task
Extract all named entities from the given text.

## Entity Types
{entity_definitions}

## Output Format
Return a valid JSON array. Each entity should have:
- "text": The exact text span from the input
- "type": One of the entity types above
- "confidence": Your confidence (high/medium/low)

## Important Rules
1. Only extract entities that are explicitly mentioned in the text
2. Do not infer or guess entities
3. Use the exact text span from the input
4. If no entities found, return an empty array []

## Input Text
{text}

## Extracted Entities (JSON)
"""

# Example usage
entity_definitions = """
- PERSON: Names of individuals (e.g., "John Smith", "Dr. Sarah Johnson")
- ORGANIZATION: Companies, agencies, institutions (e.g., "Google", "FDA")
- LOCATION: Geographic locations (e.g., "Paris", "Mount Everest")
- PRODUCT: Products, devices, services (e.g., "iPhone 15", "ChatGPT")
"""

sample_text = "Elon Musk announced that Tesla will launch the Cybertruck in Austin next month."

formatted_prompt = STRUCTURED_PROMPT.format(
    entity_definitions=entity_definitions,
    text=sample_text
)

print("📋 Formatted Prompt Example:\n")
print(formatted_prompt)

---

## 3. Using OpenAI GPT-4

### Setup

In [ ]:
# OpenAI setup (requires API key)
# Set your API key: export OPENAI_API_KEY="your-key"

import os

# Check if API key is available
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ OpenAI client initialized")
else:
    print("⚠️ OPENAI_API_KEY not found. Set it to use GPT-4.")
    print("   export OPENAI_API_KEY='your-key'")
    client = None

In [ ]:
# GPT-4 NER function
def extract_entities_gpt4(
    text: str,
    entity_types: List[str],
    model: str = "gpt-4o-mini",
    temperature: float = 0.0
) -> List[Dict]:
    """
    Extract entities using GPT-4.
    
    Args:
        text: Input text
        entity_types: List of entity types to extract
        model: GPT model to use
        temperature: Sampling temperature (0 for deterministic)
    
    Returns:
        List of entity dictionaries
    """
    if not client:
        return [{"error": "OpenAI client not initialized"}]
    
    # Build prompt
    system_prompt = """You are an expert NER system. Extract entities and return valid JSON only.
Output format: [{"text": "entity text", "type": "ENTITY_TYPE"}]
If no entities found, return: []"""
    
    user_prompt = f"""Extract these entity types: {', '.join(entity_types)}

Text: {text}

JSON output:"""
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            response_format={"type": "json_object"} if "gpt-4" in model else None
        )
        
        content = response.choices[0].message.content
        
        # Parse JSON
        result = json.loads(content)
        
        # Handle different response formats
        if isinstance(result, dict) and "entities" in result:
            return result["entities"]
        elif isinstance(result, list):
            return result
        else:
            return []
            
    except json.JSONDecodeError:
        # Try to extract JSON from response
        match = re.search(r'\[.*\]', content, re.DOTALL)
        if match:
            return json.loads(match.group())
        return []
    except Exception as e:
        return [{"error": str(e)}]

# Test with GPT-4
if client:
    test_text = "Microsoft CEO Satya Nadella announced Azure AI updates in Seattle on December 5, 2024."
    
    entities = extract_entities_gpt4(
        test_text,
        entity_types=["PERSON", "ORGANIZATION", "LOCATION", "DATE", "PRODUCT"]
    )
    
    print("🤖 GPT-4 NER Results:\n")
    print(f"Text: {test_text}\n")
    for entity in entities:
        print(f"   • '{entity.get('text', 'N/A')}' → {entity.get('type', 'N/A')}")
else:
    print("⚠️ Skipping GPT-4 test (no API key)")

---

## 4. Using Anthropic Claude

### Claude Characteristics for NER

| Feature | Behavior |
|---------|----------|
| **Consistency** | Same result for temperature=0 |
| **Following instructions** | Excellent at structured output |
| **Reasoning** | Can explain entity decisions |

In [ ]:
# Claude setup
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    from anthropic import Anthropic
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    print("✅ Anthropic client initialized")
else:
    print("⚠️ ANTHROPIC_API_KEY not found.")
    anthropic_client = None

In [ ]:
# Claude NER function
def extract_entities_claude(
    text: str,
    entity_types: List[str],
    model: str = "claude-3-haiku-20240307",
) -> List[Dict]:
    """
    Extract entities using Claude.
    """
    if not anthropic_client:
        return [{"error": "Anthropic client not initialized"}]
    
    prompt = f"""Extract named entities from the text below.

Entity types to extract: {', '.join(entity_types)}

Text: {text}

Return ONLY a valid JSON array with objects containing "text" and "type" fields.
Example format: [{{"text": "Apple", "type": "ORGANIZATION"}}]
If no entities, return: []

JSON output:"""
    
    try:
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response.content[0].text
        
        # Parse JSON
        match = re.search(r'\[.*\]', content, re.DOTALL)
        if match:
            return json.loads(match.group())
        return []
        
    except Exception as e:
        return [{"error": str(e)}]

# Test with Claude
if anthropic_client:
    test_text = "Dr. Sarah Johnson presented her research at Harvard Medical School in Boston."
    
    entities = extract_entities_claude(
        test_text,
        entity_types=["PERSON", "ORGANIZATION", "LOCATION"]
    )
    
    print("🤖 Claude NER Results:\n")
    print(f"Text: {test_text}\n")
    for entity in entities:
        print(f"   • '{entity.get('text', 'N/A')}' → {entity.get('type', 'N/A')}")
else:
    print("⚠️ Skipping Claude test (no API key)")

---

## 5. Using Open-Source LLMs (Llama, Mistral)

### Local Inference with Transformers

In [ ]:
# Using smaller open-source models for NER
# Note: Larger models like Llama-3 70B require significant GPU memory

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# For demo, we'll use a smaller model
# In production, use larger models like Llama-3-70B or Mistral-7B

print("\n⚠️ Note: For best results with open-source LLMs:")
print("   - Use Llama-3 8B/70B or Mistral-7B")
print("   - Requires GPU with sufficient VRAM")
print("   - Consider using vLLM or TGI for production")

In [ ]:
# Template for open-source LLM NER
def create_llm_ner_prompt(text: str, entity_types: List[str]) -> str:
    """
    Create a prompt for open-source LLMs.
    """
    return f"""<|system|>
You are an expert Named Entity Recognition system. Extract entities and return valid JSON.
</|system|>

<|user|>
Extract {', '.join(entity_types)} entities from this text:

"{text}"

Return JSON array: [{{"text": "entity", "type": "TYPE"}}]
</|user|>

<|assistant|>
"""

# Example for use with Ollama or local deployment
ollama_example = """
# Using Ollama for local LLM NER:

import requests

def extract_entities_ollama(text, entity_types, model="llama3"):
    prompt = f'''Extract {', '.join(entity_types)} from: "{text}"
    Return JSON: [{{"text": "entity", "type": "TYPE"}}]'''
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return json.loads(response.json()["response"])

# Usage:
# entities = extract_entities_ollama("Apple CEO Tim Cook...", ["PERSON", "ORG"])
"""

print("📝 Example Ollama Integration:")
print(ollama_example)

---

## 6. Handling LLM Challenges

### Challenge 1: Hallucination

In [ ]:
# GPT-NER approach: Mark entities in original text
# This reduces hallucination by forcing output to be derived from input

GPT_NER_MARKING_PROMPT = """
Task: Mark all {entity_type} entities in the text by surrounding them with @@ and ##.

Example:
Input: "Steve Jobs founded Apple"
Output for PERSON: "@@Steve Jobs## founded Apple"
Output for ORGANIZATION: "Steve Jobs founded @@Apple##"

Input: "{text}"
Output for {entity_type}:
"""

def extract_marked_entities(marked_text: str, entity_type: str) -> List[Dict]:
    """
    Extract entities from marked text (@@entity##).
    """
    entities = []
    pattern = r'@@(.*?)##'
    
    for match in re.finditer(pattern, marked_text):
        entities.append({
            'text': match.group(1),
            'type': entity_type,
            'start': match.start(),
            'end': match.end()
        })
    
    return entities

# Example
marked = "@@Elon Musk## announced that @@Tesla## will launch in @@Austin##."
print("📍 Parsing Marked Text:\n")
print(f"Input: {marked}\n")

entities = extract_marked_entities(marked, "ENTITY")
for ent in entities:
    print(f"   • '{ent['text']}' at position {ent['start']}-{ent['end']}")

In [ ]:
# Challenge 2: Self-verification to reduce hallucination

VERIFICATION_PROMPT = """
You extracted "{entity}" as a {entity_type} from the text.

Text: "{text}"

Questions:
1. Is "{entity}" explicitly mentioned in the text? (yes/no)
2. Is "{entity}" actually a {entity_type}? (yes/no)

If both answers are "yes", respond: VALID
Otherwise, respond: INVALID

Answer:
"""

def verify_entity(text: str, entity: str, entity_type: str) -> bool:
    """
    Verify that an extracted entity is valid.
    """
    # Simple heuristic: check if entity text exists in original
    if entity.lower() not in text.lower():
        return False
    
    # For LLM verification, you would call the API here
    # response = client.chat.completions.create(...)
    # return "VALID" in response
    
    return True

# Example usage
text = "Apple announced new products."
entities_to_verify = [
    ("Apple", "ORGANIZATION"),
    ("Microsoft", "ORGANIZATION"),  # Hallucinated!
]

print("✓ Entity Verification:\n")
print(f"Text: {text}\n")

for entity, etype in entities_to_verify:
    is_valid = verify_entity(text, entity, etype)
    status = "✅ Valid" if is_valid else "❌ Invalid (hallucination)"
    print(f"   • '{entity}' ({etype}): {status}")

In [ ]:
# Challenge 3: Consistency (temperature control)

print("🎲 Temperature Effects on NER:\n")

temperature_effects = {
    0.0: "Deterministic - Same output every time (recommended for NER)",
    0.3: "Low variation - Minor differences possible",
    0.7: "Moderate variation - Creative but less consistent",
    1.0: "High variation - Very creative, inconsistent for NER"
}

for temp, description in temperature_effects.items():
    print(f"   Temperature {temp}: {description}")

print("\n💡 Best Practice: Use temperature=0 for NER tasks")

---

## 7. LLM vs Specialized Models

In [ ]:
# Comparison table
comparison_data = {
    "Approach": ["GPT-4", "Claude 3.5", "Llama 3 70B", "GLiNER", "Fine-tuned BERT", "SpanMarker"],
    "Zero-shot": ["✅", "✅", "✅", "✅", "❌", "❌"],
    "Custom Types": ["✅", "✅", "✅", "✅", "❌", "❌"],
    "CoNLL03 F1": ["~75%", "~73%", "~70%", "~86%", "~93%", "~93%"],
    "Speed": ["Slow", "Slow", "Medium", "Fast", "Fast", "Fast"],
    "Cost": ["High", "High", "Low*", "Free", "Free", "Free"],
    "Local Deploy": ["❌", "❌", "✅", "✅", "✅", "✅"],
}

import pandas as pd
df = pd.DataFrame(comparison_data)

print("📊 LLM vs Specialized Models Comparison\n")
print(df.to_string(index=False))
print("\n* Llama cost is compute/GPU, not API")

In [ ]:
# Hybrid pipeline: Use LLM for edge cases

class HybridNERPipeline:
    """
    Hybrid pipeline that uses specialized models by default
    and falls back to LLM for difficult cases.
    """
    
    def __init__(self, specialized_model, llm_extractor, confidence_threshold=0.7):
        self.specialized = specialized_model
        self.llm = llm_extractor
        self.threshold = confidence_threshold
    
    def extract(self, text: str, entity_types: List[str]) -> List[Dict]:
        """
        Extract entities using hybrid approach.
        """
        # First, use specialized model
        entities = self.specialized.predict(text, entity_types)
        
        # Check confidence
        high_confidence = [e for e in entities if e.get('score', 0) >= self.threshold]
        low_confidence = [e for e in entities if e.get('score', 0) < self.threshold]
        
        # If low confidence entities exist, verify with LLM
        if low_confidence:
            llm_entities = self.llm(text, entity_types)
            
            # Merge: keep high confidence + LLM-verified
            verified = []
            for entity in low_confidence:
                # Check if LLM also found this entity
                llm_match = any(
                    e['text'].lower() == entity['text'].lower() 
                    for e in llm_entities
                )
                if llm_match:
                    entity['verified_by_llm'] = True
                    verified.append(entity)
            
            high_confidence.extend(verified)
        
        return high_confidence

print("📋 Hybrid Pipeline Pattern:")
print("""1. Run specialized model (GLiNER, SpanMarker)
2. Check entity confidence scores
3. For low-confidence entities, verify with LLM
4. Merge results

Benefits:
- Speed of specialized models (90%+ of cases)
- Accuracy of LLMs for edge cases
- Cost effective (minimal LLM calls)""")

---

## 8. Further Reading

### 📚 Research Papers

1. **GPT-NER: Named Entity Recognition via Large Language Models** (2023)
   - [arXiv:2304.10428](https://arxiv.org/abs/2304.10428)

2. **Recent Advances in NER: A Comprehensive Survey** (2024)
   - [arXiv:2401.10825](https://arxiv.org/html/2401.10825v3)

3. **Comparing NER with GPT4, Claude, and Mistral**
   - [Blog Post](https://unimatrixz.com/blog/talkative-ai-ner-mistral-vs-gpt-vs-claude/)

### 🔗 Resources

- [OpenAI API Documentation](https://platform.openai.com/docs/)
- [Anthropic Claude API](https://docs.anthropic.com/)
- [Ollama for Local LLMs](https://ollama.ai/)
- [vLLM for Production](https://vllm.ai/)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **LLMs for NER**: Capabilities and limitations
2. **Prompt Engineering**: Designing effective prompts
3. **GPT-4 & Claude**: API integration examples
4. **Open-Source LLMs**: Llama, Mistral options
5. **Challenges**: Hallucination, consistency, verification
6. **Hybrid Pipelines**: Best of both worlds

In [ ]:
print("🎉 Congratulations! You've completed Lesson 9: LLM-based NER")
print("\n📝 Key takeaways:")
print("   1. LLMs offer zero-shot flexibility but lower accuracy")
print("   2. Use temperature=0 for consistent results")
print("   3. GPT-NER marking reduces hallucination")
print("   4. Specialized models outperform LLMs on benchmarks")
print("   5. Hybrid pipelines combine the best of both")
print("\n👉 Continue to Lesson 10: UniversalNER/UniNER")